# OmniOpinion: Two-Timescale Conviction vs. Current Opinion Dynamics in a 100k City ($N = 100{,}000$, $T = 1{,}500$ Steps)

This notebook analyzes the **two-timescale attitude dynamics** in a medium-sized city ($N = 100{,}000$ citizens, 20 districts, $M=3$ tripolar traditional press + multi-platform social media) over $T = 1{,}500$ steps:

1. **Convictions ($\mathbf{s}_i(t) \in [-1, 1]^4$) vs. Current Opinions ($\mathbf{x}_i(t) \in [-1, 1]^4$):**
   - **Convictions** represent deep ideological anchors (`innate_beliefs`) that evolve slowly ($\eta_{\mathrm{conv}} \ll \eta_{\mathrm{op}}$) via exponential crystallization ($\tau_c = 280$ steps) and demographic inertia.
   - **Current Opinions** represent day-to-day expressed attitudes (`opinions`) that directly drive electoral voting (`ElectoralEngine`) and respond rapidly to peer interactions, media narratives, and objective news facts, while remaining **elastically tethered** within a citizen-specific radius $\Delta_{\max, i}$ around their conviction $\mathbf{s}_i(t)$.
2. **Conviction Stabilization vs. Electoral Opinion Mobility:**
   - Within the 100k city, convictions crystallize and stabilize (`conviction_step_volatility` $\to 0$), whereas current opinions continue to fluctuate (`opinion_step_volatility` $\gg$ `conviction_step_volatility`) within the conviction tether (`mean_op_conv_gap`), causing a substantial fraction of citizens (`swing_voter_fraction`) to vote for a different candidate than their underlying conviction alone would predict (`election_flipped_by_opinion`).
3. **Dependence on Citizen Features & Centrist vs. Extremist Convictions:**
   - Breakdown across **Age cohorts** (`Young <30`, `Mid 30-54`, `Senior >=55`), **Socioeconomic/Educational Status** (`Low SES <0.35`, `High SES >=0.55`), and **Conviction Extremity** ($E_i = \|\mathbf{s}_i\|_2 / \sqrt{D}$: `Centrist <0.35`, `Moderate 0.35-0.60`, `Extremist >0.60`).
   - Comparison between a **Centrist Conviction City** (`centrist_convictions`), a **Uniform Baseline City** (`uniform_random`), and an **Extremist Conviction City** (`extremist_convictions`).
4. **Strong Regime Shift in the Bias of Exogenous Facts (`fact_bias_mode = "regime_shift"`):**
   - Phase 1 ($t \in [0, 400]$): Neutral objective facts ($\mathbf{b}_{\mathrm{fact}} = 0.0$).
   - Phase 2 ($t \in [401, 950]$): Strong positive shift in fact bias ($b_{\mathrm{fact}, 0} = +0.78$).
   - Phase 3 ($t \in [951, 1500]$): Strong negative reversal in fact bias ($b_{\mathrm{fact}, 0} = -0.82$).
   - Demonstrates two-timescale hysteresis: current opinions $\bar{x}_0(t)$ shift rapidly with the biased facts and flip election winners, whereas convictions $\bar{s}_0(t)$ drift much more slowly.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

SCRATCH_CONSOLIDATED = Path('/scratch/global/crcosta/omni_opinion/consolidated/all_conviction_dynamics_metrics.parquet')
LOCAL_CONSOLIDATED = RESULTS_DIR / 'all_conviction_dynamics_metrics.parquet'
data_path = SCRATCH_CONSOLIDATED if SCRATCH_CONSOLIDATED.exists() else LOCAL_CONSOLIDATED

df = pd.read_parquet(data_path)
if 'scenario_name' not in df.columns and 'scenario' in df.columns:
    df['scenario_name'] = df['scenario']
print(f'Loaded conviction-opinion dataset from {data_path.name}: {len(df):,} rows')
print('Scenarios:', sorted(df['scenario_name'].unique()))
print('Steps range:', df['step'].min(), 'to', df['step'].max(), '| Seeds per scenario:', df.groupby('scenario_name')['task_id'].nunique().to_dict())

Loaded conviction-opinion dataset from all_conviction_dynamics_metrics.parquet: 10,836 rows
Scenarios: ['conv_centrist_factshift', 'conv_centrist_neutral', 'conv_extremist_factshift', 'conv_extremist_neutral', 'conv_uniform_factshift', 'conv_uniform_neutral']
Steps range: 0 to 1500 | Seeds per scenario: {'conv_centrist_factshift': 6, 'conv_centrist_neutral': 6, 'conv_extremist_factshift': 6, 'conv_extremist_neutral': 6, 'conv_uniform_factshift': 6, 'conv_uniform_neutral': 6}


## 1. How Convictions Stabilize While Current Opinions Fluctuate and Alter Elections

We first inspect the three neutral-fact city baselines (`conv_uniform_neutral`, `conv_centrist_neutral`, `conv_extremist_neutral`) to compare:
- **Step-to-step volatility:** `opinion_step_volatility` ($\frac{1}{N}\sum_i \|\mathbf{x}_i(t) - \mathbf{x}_i(t-1)\|_2 / \sqrt{D}$) vs. `conviction_step_volatility` ($\frac{1}{N}\sum_i \|\mathbf{s}_i(t) - \mathbf{s}_i(t-1)\|_2 / \sqrt{D}$).
- **Bounded opinion-conviction gap:** `mean_op_conv_gap` ($\frac{1}{N}\sum_i \|\mathbf{x}_i(t) - \mathbf{s}_i(t)\|_2 / \sqrt{D}$), showing that opinions move dynamically around convictions without breaking their tether.
- **Dispersion stabilization:** `conv_dispersion_ratio` stabilizing as convictions crystallize while `dispersion_ratio_vs_random` reflects active opinion clustering.
- **Electoral impact (`swing_voter_fraction` and `election_flipped_by_opinion`):** The percentage of voters whose current opinion ballot (`popular_winner`) differs from their underlying conviction ballot (`conv_popular_winner`).

In [2]:
neutral_scens = ['conv_uniform_neutral', 'conv_centrist_neutral', 'conv_extremist_neutral']
df_neutral = df[df['scenario_name'].isin(neutral_scens)].copy()

scen_labels = {
    'conv_uniform_neutral': 'Uniform City (Baseline)',
    'conv_centrist_neutral': 'Centrist Conviction City',
    'conv_extremist_neutral': 'Extremist Conviction City',
}
palette_city = {
    'conv_uniform_neutral': '#1f77b4',
    'conv_centrist_neutral': '#2ca02c',
    'conv_extremist_neutral': '#d62728',
}

final_neutral = df_neutral[df_neutral['step'] == df_neutral['step'].max()]
summary_cols = [
    'opinion_step_volatility',
    'conviction_step_volatility',
    'mean_op_conv_gap',
    'conv_dispersion_ratio',
    'dispersion_ratio_vs_random',
    'swing_voter_fraction',
    'election_flipped_by_opinion',
]
summary_table = final_neutral.groupby('scenario_name')[summary_cols].mean().round(4)
summary_table['vol_ratio_op_to_conv'] = (
    summary_table['opinion_step_volatility'] / summary_table['conviction_step_volatility'].clip(lower=1e-6)
).round(1)
print('=== Final-Step (t=1500) Conviction Stabilization vs. Opinion Mobility Summary ===')
print(summary_table.to_string())

=== Final-Step (t=1500) Conviction Stabilization vs. Opinion Mobility Summary ===
                        opinion_step_volatility  conviction_step_volatility  mean_op_conv_gap  conv_dispersion_ratio  dispersion_ratio_vs_random  swing_voter_fraction  election_flipped_by_opinion  vol_ratio_op_to_conv
scenario_name                                                                                                                                                                                                            
conv_centrist_neutral                    0.0078                      0.0000            0.1091                 0.0304                      0.0063                0.1173                       0.0000                7800.0
conv_extremist_neutral                   0.0096                      0.0000            0.0890                 1.2172                      1.1817                0.0812                       0.1667                9600.0
conv_uniform_neutral                     0.008

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10.5), sharex=True)

# Panel A: Step-to-Step Volatility (Opinion vs Conviction in log/linear scale)
ax = axes[0, 0]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')[['opinion_step_volatility', 'conviction_step_volatility']].mean()
    ax.plot(grp.index, grp['opinion_step_volatility'], color=palette_city[scen], lw=2.2,
            label=f"{scen_labels[scen]} — Current Opinion $\\Delta x$")
    ax.plot(grp.index, grp['conviction_step_volatility'], color=palette_city[scen], lw=2.0, ls='--',
            label=f"{scen_labels[scen]} — Conviction $\\Delta s$")
ax.set_yscale('log')
ax.set_title('A. Two-Timescale Step Volatility: Convictions Crystallize While Opinions Fluctuate', fontweight='bold')
ax.set_ylabel('Mean Step Volatility (Log Scale)')
ax.legend(fontsize=8.5, ncol=1, loc='upper right')

# Panel B: Bounded Opinion-Conviction Excursion Gap ||x_i(t) - s_i(t)||
ax = axes[0, 1]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')['mean_op_conv_gap'].agg(['mean', 'std'])
    ax.plot(grp.index, grp['mean'], color=palette_city[scen], lw=2.2, label=scen_labels[scen])
    ax.fill_between(grp.index, grp['mean'] - grp['std'], grp['mean'] + grp['std'], color=palette_city[scen], alpha=0.18)
ax.set_title('B. Bounded Excursion Gap Between Current Opinion and Conviction ($\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$)', fontweight='bold')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9.5, loc='lower right')

# Panel C: Conviction Dispersion vs Current Opinion Dispersion Ratio
ax = axes[1, 0]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')[['conv_dispersion_ratio', 'dispersion_ratio_vs_random']].mean()
    ax.plot(grp.index, grp['conv_dispersion_ratio'], color=palette_city[scen], lw=2.2, ls='--',
            label=f"{scen_labels[scen]} — Conviction Dispersion")
    ax.plot(grp.index, grp['dispersion_ratio_vs_random'], color=palette_city[scen], lw=2.2,
            label=f"{scen_labels[scen]} — Opinion Dispersion")
ax.set_title('C. Structural Stabilization of Convictions vs. Active Clustering of Opinions', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Dispersion Ratio vs. Uniform Random')
ax.legend(fontsize=8.5, loc='upper right')

# Panel D: Electoral Swing Voters Driven by Opinion Mobility Around Convictions
ax = axes[1, 1]
for scen in neutral_scens:
    sub = df_neutral[df_neutral['scenario_name'] == scen]
    grp = sub.groupby('step')['swing_voter_fraction'].agg(['mean', 'std'])
    ax.plot(grp.index, 100.0 * grp['mean'], color=palette_city[scen], lw=2.3, label=scen_labels[scen])
    ax.fill_between(grp.index, 100.0 * (grp['mean'] - grp['std']), 100.0 * (grp['mean'] + grp['std']),
                    color=palette_city[scen], alpha=0.18)
ax.set_title('D. Electoral Impact: Voters Casting a Different Ballot Than Their Conviction Alone', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Swing Voter Percentage (%)')
ax.legend(fontsize=9.5, loc='lower right')

plt.tight_layout()
fig1_path = FIGURES_DIR / 'fig_cv1_conviction_stabilization_vs_opinion_mobility.png'
fig.savefig(fig1_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig1_path}')

<Figure size 2100x1470 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv1_conviction_stabilization_vs_opinion_mobility.png


## 2. How Opinion Mobility & Electoral Swing Depend on Person Features and Centrist vs. Extremist Convictions

Next, we examine how a citizen's demographic and ideological attributes shape the interplay between conviction and current opinion:
- **Age Cohorts (Life-Stage Plasticity vs. Crystallized Attitude Persistence):**
  - `Young (<30)`: Higher social media exposure and openness, wider conviction tether $\Delta_{\max, i}$, larger opinion excursion, and higher electoral swing rate.
  - `Mid-Age (30–54)`: Intermediate tether and swing rate.
  - `Senior (>=55)`: Strongly crystallized convictions, tighter tether $\Delta_{\max, i}$, and lower opinion volatility.
- **Socioeconomic / Educational Status (SES):**
  - `Low SES (<0.35)` vs. `High SES (>=0.55)`: Higher cognitive/educational resources provide stronger ideological constraint, keeping daily opinions closer to underlying convictions.
- **Conviction Extremity ($E_i = \|\mathbf{s}_i\|_2 / \sqrt{D}$):**
  - `Centrist Conviction (<0.35)`: Weak ego-involvement allows wider opinion mobility (`centrist_op_conv_gap`) and makes centrist citizens the decisive electoral swing group (`centrist_swing_vote`).
  - `Moderate Conviction (0.35–0.60)`: Intermediate mobility.
  - `Extremist Conviction (>0.60)`: High ego-involvement (Sherif & Hovland, 1961) strongly anchors daily opinions near the extreme conviction (`extremist_op_conv_gap`) and virtually locks in their vote (`extremist_swing_vote` is minimal).

In [4]:
steady_df = df_neutral[df_neutral['step'] >= 500].copy()
uni_steady = steady_df[steady_df['scenario_name'] == 'conv_uniform_neutral']

cohort_table = pd.DataFrame({
    'Group Category': [
        'Age: Young (<30)', 'Age: Mid (30-54)', 'Age: Senior (>=55)',
        'SES: Low (<0.35)', 'SES: High (>=0.55)',
        'Conviction: Centrist (<0.35)', 'Conviction: Moderate (0.35-0.60)', 'Conviction: Extremist (>0.60)'
    ],
    'Mean Opinion-Conviction Gap': [
        uni_steady['op_conv_gap_young'].mean(),
        uni_steady['op_conv_gap_mid'].mean(),
        uni_steady['op_conv_gap_senior'].mean(),
        uni_steady['op_conv_gap_low_ses'].mean(),
        uni_steady['op_conv_gap_high_ses'].mean(),
        uni_steady['op_conv_gap_centrist'].mean(),
        uni_steady['op_conv_gap_moderate'].mean(),
        uni_steady['op_conv_gap_extremist'].mean(),
    ],
    'Mean Opinion Step Volatility': [
        uni_steady['op_vol_young'].mean(),
        uni_steady['op_vol_mid'].mean(),
        uni_steady['op_vol_senior'].mean(),
        uni_steady['op_vol_low_ses'].mean(),
        uni_steady['op_vol_high_ses'].mean(),
        uni_steady['op_vol_centrist'].mean(),
        uni_steady['op_vol_moderate'].mean(),
        uni_steady['op_vol_extremist'].mean(),
    ],
    'Swing Vote Rate (%)': [
        100.0 * uni_steady['swing_vote_young'].mean(),
        100.0 * uni_steady['swing_vote_mid'].mean(),
        100.0 * uni_steady['swing_vote_senior'].mean(),
        100.0 * uni_steady['swing_vote_low_ses'].mean(),
        100.0 * uni_steady['swing_vote_high_ses'].mean(),
        100.0 * uni_steady['swing_vote_centrist'].mean(),
        100.0 * uni_steady['swing_vote_moderate'].mean(),
        100.0 * uni_steady['swing_vote_extremist'].mean(),
    ],
}).round(4)

print('=== Citizen Feature & Conviction Extremity Breakdown (Steady State t in [500, 1500]) ===')
print(cohort_table.to_string(index=False))

=== Citizen Feature & Conviction Extremity Breakdown (Steady State t in [500, 1500]) ===
                  Group Category  Mean Opinion-Conviction Gap  Mean Opinion Step Volatility  Swing Vote Rate (%)
                Age: Young (<30)                       0.1719                        0.0079              22.1197
                Age: Mid (30-54)                       0.1557                        0.0078              17.9988
              Age: Senior (>=55)                       0.1167                        0.0079              10.8941
                SES: Low (<0.35)                       0.1485                        0.0079              17.2751
              SES: High (>=0.55)                       0.1505                        0.0079              16.9319
    Conviction: Centrist (<0.35)                       0.1815                        0.0077              26.5742
Conviction: Moderate (0.35-0.60)                       0.1522                        0.0077              13.1141
   Conv

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.5))

# Panel A: Time series of Opinion-Conviction Gap by Age & SES
ax = axes[0, 0]
uni_all = df_neutral[df_neutral['scenario_name'] == 'conv_uniform_neutral']
age_grp = uni_all.groupby('step')[[
    'op_conv_gap_young', 'op_conv_gap_mid', 'op_conv_gap_senior',
    'op_conv_gap_low_ses', 'op_conv_gap_high_ses'
]].mean()
ax.plot(age_grp.index, age_grp['op_conv_gap_young'], color='#ff7f0e', lw=2.3, label='Age: Young (<30)')
ax.plot(age_grp.index, age_grp['op_conv_gap_mid'], color='#1f77b4', lw=2.1, label='Age: Mid (30–54)')
ax.plot(age_grp.index, age_grp['op_conv_gap_senior'], color='#9467bd', lw=2.3, label='Age: Senior (>=55)')
ax.plot(age_grp.index, age_grp['op_conv_gap_low_ses'], color='#d62728', lw=1.9, ls='--', label='SES: Low (<0.35)')
ax.plot(age_grp.index, age_grp['op_conv_gap_high_ses'], color='#2ca02c', lw=1.9, ls='--', label='SES: High (>=0.55)')
ax.set_title('A. Opinion Excursion ($\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$) by Age Cohort and SES', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9, ncol=2, loc='lower right')

# Panel B: Time series of Opinion-Conviction Gap by Conviction Extremity (Centrist vs Moderate vs Extremist)
ax = axes[0, 1]
ext_grp = uni_all.groupby('step')[['op_conv_gap_centrist', 'op_conv_gap_moderate', 'op_conv_gap_extremist']].mean()
ax.plot(ext_grp.index, ext_grp['op_conv_gap_centrist'], color='#2ca02c', lw=2.5, label='Centrist Conviction ($E_i < 0.35$)')
ax.plot(ext_grp.index, ext_grp['op_conv_gap_moderate'], color='#1f77b4', lw=2.3, label='Moderate Conviction ($0.35 \\leq E_i \\leq 0.60$)')
ax.plot(ext_grp.index, ext_grp['op_conv_gap_extremist'], color='#d62728', lw=2.5, label='Extremist Conviction ($E_i > 0.60$)')
ax.set_title('B. Opinion Mobility Within Conviction Tether: Centrist vs. Extremist Citizens', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Opinion–Conviction Distance')
ax.legend(fontsize=9.5, loc='lower right')

# Panel C: Bar Chart of Electoral Swing Rate (%) Across Citizen Subgroups
ax = axes[1, 0]
subgroups = [
    'Young\n(<30)', 'Mid-Age\n(30–54)', 'Senior\n(>=55)',
    'Low SES\n(<0.35)', 'High SES\n(>=0.55)',
    'Centrist\nConviction', 'Moderate\nConviction', 'Extremist\nConviction'
]
swing_vals = [
    100.0 * uni_steady['swing_vote_young'].mean(),
    100.0 * uni_steady['swing_vote_mid'].mean(),
    100.0 * uni_steady['swing_vote_senior'].mean(),
    100.0 * uni_steady['swing_vote_low_ses'].mean(),
    100.0 * uni_steady['swing_vote_high_ses'].mean(),
    100.0 * uni_steady['swing_vote_centrist'].mean(),
    100.0 * uni_steady['swing_vote_moderate'].mean(),
    100.0 * uni_steady['swing_vote_extremist'].mean(),
]
bar_colors = ['#ff7f0e', '#1f77b4', '#9467bd', '#e377c2', '#17becf', '#2ca02c', '#7f7f7f', '#d62728']
bars = ax.bar(subgroups, swing_vals, color=bar_colors, edgecolor='black', alpha=0.88)
for b, v in zip(bars, swing_vals):
    ax.text(b.get_x() + b.get_width() / 2.0, v + 0.35, f'{v:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(swing_vals) * 1.18)
ax.set_title('C. Electoral Swing Rate (% Voting Differently Than Conviction) by Citizen Feature', fontweight='bold')
ax.set_ylabel('Swing Voters (%)')

# Panel D: City-Level Comparison (Centrist City vs Uniform City vs Extremist City)
ax = axes[1, 1]
city_metrics = steady_df.groupby('scenario_name')[[
    'mean_op_conv_gap', 'swing_voter_fraction', 'election_flipped_by_opinion'
]].mean().loc[neutral_scens]
x = np.arange(len(neutral_scens))
w = 0.26
b1 = ax.bar(x - w, 100.0 * city_metrics['mean_op_conv_gap'], width=w, color='#1f77b4', edgecolor='black',
            label='Mean Gap $\\|\\mathbf{x}-\\mathbf{s}\\| \\times 100$')
b2 = ax.bar(x, 100.0 * city_metrics['swing_voter_fraction'], width=w, color='#2ca02c', edgecolor='black',
            label='Swing Voter Share (%)')
b3 = ax.bar(x + w, 100.0 * city_metrics['election_flipped_by_opinion'], width=w, color='#d62728', edgecolor='black',
            label='Election Flipped by Opinion (%)')
ax.set_xticks(x)
ax.set_xticklabels(['Uniform City\n(Baseline)', 'Centrist\nConviction City', 'Extremist\nConviction City'])
ax.set_title('D. Macro-City Comparison: Centrist Conviction City vs. Extremist Conviction City', fontweight='bold')
ax.set_ylabel('Percentage / Scaled Index')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
fig2_path = FIGURES_DIR / 'fig_cv2_person_features_and_centrist_vs_extremist.png'
fig.savefig(fig2_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig2_path}')

<Figure size 2170x1470 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv2_person_features_and_centrist_vs_extremist.png


## 3. Strong Shift in the Bias of Exogenous Facts: Fast Opinion Response vs. Slow Conviction Drift

Finally, we analyze the **Fact Bias Regime-Shift Experiment (`fact_bias_mode = "regime_shift"`)** across all three city architectures (`conv_uniform_factshift`, `conv_centrist_factshift`, `conv_extremist_factshift`):
- **Phase 1 ($t \in [0, 400]$):** Neutral exogenous facts ($b_{\mathrm{fact}, 0} = 0.0$).
- **Phase 2 ($t \in [401, 950]$):** Strong positive shift in exogenous fact bias ($b_{\mathrm{fact}, 0} = +0.78, b_{\mathrm{fact}, 1} = +0.58$).
- **Phase 3 ($t \in [951, 1500]$):** Strong negative reversal in exogenous fact bias ($b_{\mathrm{fact}, 0} = -0.82, b_{\mathrm{fact}, 1} = -0.62$).

This experiment highlights two key phenomena:
1. **Timescale Separation (Hysteresis):** Current opinions $\bar{x}_0(t)$ respond rapidly to the new factual environment, stretching toward the conviction tether limit $\Delta_{\max, i}$, while convictions $\bar{s}_0(t)$ shift slowly and smoothly (`conviction_step_volatility` $\ll$ `opinion_step_volatility`).
2. **Electoral Reversal Without Full Conviction Conversion:** During Phase 2 and Phase 3, current opinions shift enough to flip candidate vote shares (`share_cand_2` vs. `share_cand_0`) and trigger high rates of `election_flipped_by_opinion`, especially in the **Centrist Conviction City**, even though underlying convictions have moved only a fraction of the distance.

In [6]:
shift_scens = ['conv_uniform_factshift', 'conv_centrist_factshift', 'conv_extremist_factshift']
df_shift = df[df['scenario_name'].isin(shift_scens)].copy()

shift_labels = {
    'conv_uniform_factshift': 'Uniform City (Fact Shift)',
    'conv_centrist_factshift': 'Centrist City (Fact Shift)',
    'conv_extremist_factshift': 'Extremist City (Fact Shift)',
}
palette_shift = {
    'conv_uniform_factshift': '#1f77b4',
    'conv_centrist_factshift': '#2ca02c',
    'conv_extremist_factshift': '#d62728',
}

checkpoints = [400, 950, 1500]
chk_df = df_shift[df_shift['step'].isin(checkpoints)].groupby(['scenario_name', 'step'])[[
    'fact_bias_d0', 'pop_cm_d0', 'conv_cm_d0', 'mean_op_conv_gap', 'swing_voter_fraction', 'election_flipped_by_opinion'
]].mean().round(4)
print('=== Fact-Bias Regime Shift Trajectory Checkpoints (t = 400 Neutral End, t = 950 +0.78 Peak, t = 1500 -0.82 Reversal) ===')
print(chk_df.to_string())

=== Fact-Bias Regime Shift Trajectory Checkpoints (t = 400 Neutral End, t = 950 +0.78 Peak, t = 1500 -0.82 Reversal) ===
                               fact_bias_d0  pop_cm_d0  conv_cm_d0  mean_op_conv_gap  swing_voter_fraction  election_flipped_by_opinion
scenario_name            step                                                                                                          
conv_centrist_factshift  400           0.00    -0.0589     -0.0270            0.1549                0.1830                       0.0000
                         950           0.78     0.0495      0.0005            0.1207                0.1301                       0.0000
                         1500         -0.82    -0.1624     -0.0420            0.1408                0.0640                       0.0000
conv_extremist_factshift 400           0.00    -0.0222     -0.0045            0.1148                0.0872                       0.1667
                         950           0.78     0.0257      0.0

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(15.5, 11.0), sharex=True)

def shade_regimes(ax):
    ax.axvline(400, color='black', ls=':', lw=1.3, alpha=0.75)
    ax.axvline(950, color='black', ls=':', lw=1.3, alpha=0.75)
    ax.axvspan(400, 950, color='#ffbb78', alpha=0.14)
    ax.axvspan(950, 1500, color='#aec7e8', alpha=0.14)

# Panel A: Exogenous Fact Bias + Fast Opinion CM vs Slow Conviction CM (Uniform & Centrist Cities)
ax = axes[0, 0]
shade_regimes(ax)
ref_fb = df_shift[df_shift['scenario_name'] == 'conv_uniform_factshift'].groupby('step')['fact_bias_d0'].mean()
ax.plot(ref_fb.index, ref_fb.values, color='black', lw=1.6, ls=':', label='Exogenous Fact Bias $b_{\\mathrm{fact},0}(t)$')
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen]
    grp = sub.groupby('step')[['pop_cm_d0', 'conv_cm_d0']].mean()
    ax.plot(grp.index, grp['pop_cm_d0'], color=palette_shift[scen], lw=2.4,
            label=f"{shift_labels[scen]} — Current Opinion $\\bar{{x}}_0(t)$")
    ax.plot(grp.index, grp['conv_cm_d0'], color=palette_shift[scen], lw=2.1, ls='--',
            label=f"{shift_labels[scen]} — Conviction $\\bar{{s}}_0(t)$")
ax.set_title('A. Strong Shift in Fact Bias: Fast Current Opinion Response vs. Slow Conviction Drift', fontweight='bold')
ax.set_ylabel('Mean Coordinate on Primary Issue ($d=0$)')
ax.legend(fontsize=8.0, ncol=1, loc='lower left')

# Panel B: Signed Lag / Tension Between Current Opinion and Conviction (x_0 - s_0) and Bounded Gap
ax = axes[0, 1]
shade_regimes(ax)
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen].copy()
    sub['signed_lead_d0'] = sub['pop_cm_d0'] - sub['conv_cm_d0']
    grp = sub.groupby('step')[['signed_lead_d0', 'mean_op_conv_gap']].mean()
    ax.plot(grp.index, grp['signed_lead_d0'], color=palette_shift[scen], lw=2.3,
            label=f"{shift_labels[scen]} — Signed Lead ($\\bar{{x}}_0 - \\bar{{s}}_0$)")
    ax.plot(grp.index, grp['mean_op_conv_gap'], color=palette_shift[scen], lw=1.8, ls='--',
            label=f"{shift_labels[scen]} — Total Gap $\\|\\mathbf{{x}}-\\mathbf{{s}}\\|$")
ax.axhline(0.0, color='gray', lw=1.0)
ax.set_title('B. Two-Timescale Elastic Tension: Opinion Leads Conviction During Fact Shocks', fontweight='bold')
ax.set_ylabel('Opinion–Conviction Displacement')
ax.legend(fontsize=8.3, loc='lower left')

# Panel C: Electoral Vote Shares Under Fact Regime Shift (Opinion Ballot vs Counterfactual Conviction Ballot)
ax = axes[1, 0]
shade_regimes(ax)
cent_sub = df_shift[df_shift['scenario_name'] == 'conv_centrist_factshift']
cent_grp = cent_sub.groupby('step')[[
    'share_cand_0', 'share_cand_2', 'conv_share_cand_0', 'conv_share_cand_2'
]].mean()
ax.plot(cent_grp.index, 100.0 * cent_grp['share_cand_2'], color='#d62728', lw=2.5,
        label='Cand 2 (+ Pole) — Actual Opinion Vote Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['conv_share_cand_2'], color='#d62728', lw=2.1, ls='--',
        label='Cand 2 (+ Pole) — Conviction-Only Vote Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['share_cand_0'], color='#1f77b4', lw=2.5,
        label='Cand 0 (– Pole) — Actual Opinion Vote Share')
ax.plot(cent_grp.index, 100.0 * cent_grp['conv_share_cand_0'], color='#1f77b4', lw=2.1, ls='--',
        label='Cand 0 (– Pole) — Conviction-Only Vote Share')
ax.set_title('C. Electoral Impact in Centrist City: Current Opinion Flips the Election Before Convictions Move', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Candidate Vote Share (%)')
ax.legend(fontsize=8.5, loc='upper left')

# Panel D: Swing Voter Percentage Across Fact-Bias Regimes
ax = axes[1, 1]
shade_regimes(ax)
for scen in shift_scens:
    sub = df_shift[df_shift['scenario_name'] == scen]
    grp = sub.groupby('step')['swing_voter_fraction'].agg(['mean', 'std'])
    ax.plot(grp.index, 100.0 * grp['mean'], color=palette_shift[scen], lw=2.3, label=shift_labels[scen])
    ax.fill_between(grp.index, 100.0 * (grp['mean'] - grp['std']), 100.0 * (grp['mean'] + grp['std']),
                    color=palette_shift[scen], alpha=0.16)
ax.set_title('D. Surge in Swing Voting During Strong Shifts in Fact Bias', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Swing Voters Casting Non-Conviction Ballot (%)')
ax.legend(fontsize=9.5, loc='upper left')

plt.tight_layout()
fig3_path = FIGURES_DIR / 'fig_cv3_fact_bias_regime_shift_hysteresis.png'
fig.savefig(fig3_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig3_path}')

<Figure size 2170x1540 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_cv3_fact_bias_regime_shift_hysteresis.png
